In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ==============================================================================
# 0. SAMPLE DATA GENERATION (Replace with pd.read_csv() for external datasets)
# ==============================================================================
np.random.seed(42)
n_records = 1000

neighbourhoods_data = pd.DataFrame({
    "neighbourhood_id": ["NBH_01", "NBH_02", "NBH_03", "NBH_04", "NBH_05"],
    "neighbourhood_name": [
        "Greenwood Heights", "Oakridge Park", "Riverside Quarter",
        "Maplewood Commons", "Southgate Industrial"
    ],
    "zone_type": ["Urban Core", "Suburban", "Waterfront", "Suburban", "Transitioning"],
    "transit_score": [9.2, 7.8, 8.9, 6.5, 5.4],
    "school_rating": [8.8, 9.1, 8.4, 7.6, 6.1],
    "median_income_usd": [145000, 118000, 138000, 89000, 68000],
})

nbh_choices = np.random.choice(neighbourhoods_data["neighbourhood_id"], size=n_records, p=[0.22, 0.24, 0.18, 0.20, 0.16])
prop_choices = np.random.choice(["Apartment", "apt", "Condo", "Townhouse", "Detached House", "HOUSE "], size=n_records)
record_choices = np.random.choice(["Sale", "Listing"], size=n_records, p=[0.70, 0.30])
currency_choices = np.random.choice(["USD", "EUR", "INR_LAKHS"], size=n_records, p=[0.75, 0.15, 0.10])
unit_choices = np.random.choice(["sqft", "sqm"], size=n_records, p=[0.80, 0.20])

base_areas_sqft = np.random.uniform(550, 3200, size=n_records)
raw_areas = np.where(unit_choices == "sqm", base_areas_sqft / 10.7639, base_areas_sqft)

nbh_mult_map = {"NBH_01": 1.30, "NBH_02": 1.15, "NBH_03": 1.25, "NBH_04": 0.95, "NBH_05": 0.78}
mults = np.array([nbh_mult_map[n] for n in nbh_choices])
bias_mults = np.where(record_choices == "Listing", 1.07, 1.00)  # +7% asking-price bias
usd_prices = base_areas_sqft * 330 * mults * bias_mults * np.random.lognormal(0, 0.12, size=n_records)

raw_prices = np.where(
    currency_choices == "EUR", usd_prices * 0.92,
    np.where(currency_choices == "INR_LAKHS", (usd_prices * 83.5) / 100000.0, usd_prices)
)

dates = pd.date_range("2024-01-01", "2025-12-31", freq="D")
raw_dates = [
    pd.Timestamp(d).strftime("%Y-%m-%d") if i % 5 != 0 else pd.Timestamp(d).strftime("%d/%m/%Y")
    for i, d in enumerate(np.random.choice(dates, size=n_records))
]

raw_listings = pd.DataFrame({
    "listing_id": [f"PROP_{1000 + i}" for i in range(n_records)],
    "neighbourhood_id": nbh_choices,
    "property_type": prop_choices,
    "record_type": record_choices,
    "transaction_date": raw_dates,
    "price": np.round(raw_prices, 2),
    "currency": currency_choices,
    "area": np.round(raw_areas, 2),
    "area_unit": unit_choices,
    "bedrooms": np.random.choice([1, 2, 3, 4], size=n_records),
    "has_parking": np.random.choice([0, 1], size=n_records, p=[0.25, 0.75]),
    "renovated": np.random.choice([0, 1], size=n_records, p=[0.65, 0.35]),
    "latitude": np.round(40.70 + np.random.uniform(0.01, 0.15, size=n_records), 6),
    "longitude": np.round(-74.02 + np.random.uniform(0.01, 0.15, size=n_records), 6),
})

# Inject realistic anomalies to validate cleaning pipeline
raw_listings = pd.concat([raw_listings, raw_listings.iloc[:12]], ignore_index=True)  # 12 duplicates
raw_listings.loc[3, "price"] = 10.0         # Implausible price
raw_listings.loc[8, "area"] = -120.0        # Implausible negative area
raw_listings.loc[15, "neighbourhood_id"] = "NBH_99"  # Unmatched foreign key

# ==============================================================================
# 1. STANDARDISE UNITS, CURRENCIES, DATES & PROPERTY CATEGORIES
# ==============================================================================
df = raw_listings.copy()

category_map = {
    "apt": "Apartment",
    "apartment": "Apartment",
    "condo": "Condo",
    "townhouse": "Townhouse",
    "house": "Detached House",
    "detached house": "Detached House",
}
df["property_type_clean"] = (
    df["property_type"].str.strip().str.lower().map(category_map).fillna("Other")
)
df["date_clean"] = pd.to_datetime(df["transaction_date"], format="mixed", dayfirst=True)

fx_rates = {"USD": 1.0, "EUR": 1.087, "INR_LAKHS": 1197.6}
df["price_usd"] = np.round(df["price"].to_numpy() * df["currency"].map(fx_rates).to_numpy(), 2)
df["area_sqft"] = np.round(
    np.where(df["area_unit"] == "sqm", df["area"].to_numpy() * 10.7639, df["area"].to_numpy()), 2
)

# ==============================================================================
# 2. DETECT DUPLICATE LISTINGS & IMPLAUSIBLE PRICE / AREA VALUES
# ==============================================================================
duplicates_removed = int(df.duplicated(subset=["listing_id"]).sum())
df = df.drop_duplicates(subset=["listing_id"], keep="first").copy()

plausible_mask = (
    (df["area_sqft"].between(250, 15000))
    & (df["price_usd"].between(40000, 15000000))
)
implausible_removed = int((~plausible_mask).sum())
df = df.loc[plausible_mask].copy()

# ==============================================================================
# 3. ENGINEER PRICE PER UNIT AREA & PROPERTY ATTRIBUTES
# ==============================================================================
df["price_per_sqft"] = np.round(df["price_usd"] / df["area_sqft"], 2)
df["amenity_score"] = df["has_parking"] * 1 + df["renovated"] * 2
df["size_tier"] = np.select(
    [df["area_sqft"] < 1000, df["area_sqft"].between(1000, 1999), df["area_sqft"] >= 2000],
    ["Compact (<1,000 sqft)", "Mid-Size (1,000-1,999 sqft)", "Large (2,000+ sqft)"],
    default="Mid-Size (1,000-1,999 sqft)",
)

# ==============================================================================
# 4. NUMPY ROBUST OUTLIER ANALYSIS & QUANTILE COMPARISONS
# ==============================================================================
ppsf_arr = df["price_per_sqft"].to_numpy()
q10, q25, q50, q75, q90 = np.percentile(ppsf_arr, [10, 25, 50, 75, 90])
iqr = q75 - q25
mad = np.median(np.abs(ppsf_arr - q50))
df["modified_z_score"] = np.round(0.6745 * (ppsf_arr - q50) / mad, 3)
df["is_outlier"] = (np.abs(df["modified_z_score"]) > 3.5) | (ppsf_arr > q75 + 1.5 * iqr)

# ==============================================================================
# 5 & 6. JOIN NEIGHBOURHOOD INDICATORS & COMPARE BY LOCATION, TYPE, SIZE & TIME
# ==============================================================================
assert neighbourhoods_data["neighbourhood_id"].is_unique
merged_df = pd.merge(
    df,
    neighbourhoods_data,
    on="neighbourhood_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched_rows = int((merged_df["_merge"] == "left_only").sum())
clean_df = merged_df.loc[merged_df["_merge"] == "both"].drop(columns=["_merge"]).reset_index(drop=True)

neighbourhood_comparison = (
    clean_df.groupby(["neighbourhood_name", "zone_type", "transit_score", "school_rating"])
    .agg(
        listings=("listing_id", "count"),
        median_price_usd=("price_usd", "median"),
        median_ppsf=("price_per_sqft", "median"),
        p25_ppsf=("price_per_sqft", lambda s: np.percentile(s, 25)),
        p75_ppsf=("price_per_sqft", lambda s: np.percentile(s, 75)),
    )
    .reset_index()
    .sort_values("median_ppsf", ascending=False)
)

monthly_trends = (
    clean_df.set_index("date_clean")
    .groupby("zone_type")["price_per_sqft"]
    .resample("M")
    .median()
    .reset_index()
)
monthly_trends["rolling_3m_ppsf"] = (
    monthly_trends.groupby("zone_type")["price_per_sqft"]
    .transform(lambda s: s.rolling(3, min_periods=1).mean())
    .round(2)
)

# ==============================================================================
# 7 & 8. COMPARABLE SEGMENTS, SENSITIVITY CHECKS & LISTING-VS-SALE BIAS
# ==============================================================================
clean_df["comparable_segment"] = (
    clean_df["zone_type"] + " | " + clean_df["property_type_clean"] + " | " + clean_df["size_tier"]
)
seg_median = clean_df.groupby("comparable_segment")["price_per_sqft"].transform("median")
clean_df["relative_price_index"] = np.round((clean_df["price_per_sqft"] / seg_median) * 100, 2)

bias_sensitivity = clean_df.pivot_table(
    values="price_per_sqft", index="neighbourhood_name", columns="record_type", aggfunc="median"
).reset_index()
bias_sensitivity["listing_asking_premium_pct"] = np.round(
    ((bias_sensitivity["Listing"] - bias_sensitivity["Sale"]) / bias_sensitivity["Sale"]) * 100, 2
)

inlier_sale_medians = (
    clean_df.query("not is_outlier and record_type == 'Sale'")
    .groupby("neighbourhood_name")["price_per_sqft"]
    .median()
    .rename("Sale_Without_Outliers")
)
bias_sensitivity = bias_sensitivity.merge(inlier_sale_medians, on="neighbourhood_name")
bias_sensitivity["outlier_shift_pct"] = np.round(
    ((bias_sensitivity["Sale"] - bias_sensitivity["Sale_Without_Outliers"]) / bias_sensitivity["Sale_Without_Outliers"]) * 100, 2
)

# ==============================================================================
# EXPECTED SUBMISSIONS: CLEAN TABLE, MAP-READY CSV & INSIGHT SUMMARY
# ==============================================================================
clean_df.to_csv("clean_property_table.csv", index=False)
neighbourhood_comparison.to_csv("neighbourhood_comparison.csv", index=False)
clean_df[[
    "listing_id", "neighbourhood_name", "property_type_clean", "size_tier",
    "record_type", "price_usd", "area_sqft", "price_per_sqft", "latitude", "longitude"
]].to_csv("map_ready_output.csv", index=False)

print(f"Audit -> Duplicates removed: {duplicates_removed} | Implausible removed: {implausible_removed} | Unmatched keys: {unmatched_rows} | Final rows: {len(clean_df)}")
print(f"Global PPSF Quantiles -> P10: ${q10:.2f}, Median: ${q50:.2f}, P90: ${q90:.2f}, IQR: ${iqr:.2f}")
print("\n--- Neighbourhood Comparison ---")
print(neighbourhood_comparison.to_string(index=False))
print("\n--- Listing-vs-Sale Bias & Outlier Sensitivity ---")
print(bias_sensitivity.to_string(index=False))

Audit -> Duplicates removed: 12 | Implausible removed: 2 | Unmatched keys: 1 | Final rows: 997
Global PPSF Quantiles -> P10: $266.06, Median: $380.00, P90: $473.21, IQR: $108.29

--- Neighbourhood Comparison ---
  neighbourhood_name     zone_type  transit_score  school_rating  listings  median_price_usd  median_ppsf  p25_ppsf  p75_ppsf
   Greenwood Heights    Urban Core            9.2            8.8       237        800378.870      436.180  397.9800   474.950
   Riverside Quarter    Waterfront            8.9            8.4       186        714293.285      415.275  390.1525   454.075
       Oakridge Park      Suburban            7.8            9.1       233        671951.670      380.850  357.2000   410.040
   Maplewood Commons      Suburban            6.5            7.6       182        580387.975      323.140  294.5425   349.900
Southgate Industrial Transitioning            5.4            6.1       159        506239.200      261.430  240.8350   280.620

--- Listing-vs-Sale Bias & Outl

/tmp/ipykernel_1224/2040008653.py:162: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample("M")
